# 🏦 Loan Default Prediction & Credit Risk Underwriting Pipeline
### Comprehensive Machine Learning Project aligned with Department Standard Operating Procedure (SOP)

---

## 📋 SOP Syllabus & Project Checklist Alignment:
1. **Section 1: Problem Definition & Dataset Exploration**
   - Problem statement, objective, business context, dataset summary, and data dictionary.
2. **Section 2: Data Cleaning, Preprocessing & Exploratory Data Analysis (EDA)**
   - Missing value verification & imputation.
   - Outlier detection & handling using Interquartile Range (IQR).
   - Categorical variable encoding & binary normalization.
   - Feature scaling using `StandardScaler`.
   - Comprehensive EDA (Target distribution, correlation matrix heatmap).
3. **Section 3: Model Creation (With Library & FROM SCRATCH WITHOUT LIBRARY)**
   - Mathematical foundations.
   - **Custom Scratch Implementation:** `CustomLogisticRegressionFromScratch` built using pure NumPy (Sigmoid, Binary Cross-Entropy Loss, Gradient Descent).
   - Comparative verification: Scratch Implementation vs. Scikit-learn Library Implementation.
4. **Section 4: Baseline Model Training & Overfitting / Underfitting Diagnostics**
   - Logistic Regression, Decision Tree, K-Nearest Neighbors (KNN), Random Forest (optimized depth).
   - Train vs. Test score comparison to diagnose Overfitting/Underfitting.
5. **Section 5: Advanced Model Training, Cross-Validation & Hyperparameter Tuning**
   - Gaussian Naive Bayes (Probabilistic Classifier).
   - Calibrated Support Vector Classifier (LinearSVC + CalibratedClassifierCV).
   - Bagging Classifier (Bootstrap Aggregating with balanced trees).
   - Gradient Boosting (Histogram-based Gradient Boosting) & AdaBoost.
   - K-Fold Stratified Cross-Validation for stability testing.
   - Hyperparameter Tuning using `GridSearchCV`.
6. **Section 6: Comprehensive Metrics Visualization & Performance Analysis**
   - Multi-model ROC Curves & Area Under the Curve (AUC).
   - Confusion Matrix Heatmaps.
   - Feature Importance Bar Charts.
   - Train vs. Test Generalization Gap Analysis.
7. **Section 7: Model Export & Deployment Readiness**
   - Exporting all trained models and scalers (`models/*.pkl`) for the Web Application (`app.py`).

---
# 📌 Section 1: Problem Definition & Dataset Exploration

### 1.1 Problem Statement
In retail banking and consumer lending, evaluating credit risk accurately is paramount. When a financial institution grants a loan, the borrower may fail to make required debt payments, resulting in a **Loan Default**.
- **Business Goal:** Predict the probability that a borrower will default on their loan ($Y=1$) versus repay successfully ($Y=0$).
- **Machine Learning Objective:** Build and benchmark multiple supervised binary classification models to optimize for **Recall** and **ROC-AUC**.

### 1.2 Data Dictionary
- `Age`: Age of borrower (years)
- `Income`: Annual income of borrower ($USD)
- `LoanAmount`: Total requested loan amount ($USD)
- `CreditScore`: Bureau credit score (300 to 850)
- `MonthsEmployed`: Continuous employment duration in months
- `NumCreditLines`: Number of open credit facilities
- `InterestRate`: Annualized interest rate on loan (%)
- `LoanTerm`: Duration of loan (in months)
- `DTIRatio`: Debt-to-Income ratio
- `HasMortgage`: Whether borrower holds an existing mortgage (0 = No, 1 = Yes)
- `HasDependents`: Whether borrower has dependents (0 = No, 1 = Yes)
- `LoanDefault` (Target): `0` = Non-Default, `1` = Default

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print("Environment setup and libraries loaded successfully!")

In [ ]:
data_file = 'preprocessed_loan_default.csv' if os.path.exists('preprocessed_loan_default.csv') else 'Loan_default.csv'
print(f"Loading data from: {data_file}")
df = pd.read_csv(data_file)
print(f"Dataset Dimensions: {df.shape[0]:,} Rows | {df.shape[1]} Columns")
print("\n--- First 5 Rows ---")
df.head()

In [ ]:
print("--- Statistical Summary of Features ---")
df.describe().T

---
# 🧹 Section 2: Data Cleaning, Preprocessing & EDA

1. **Missing Value Verification**: Check null counts.
2. **Categorical Encoding & Standardization**: Convert string flags (`'Yes'`/`'No'`) to binary integers (`1`/`0`).
3. **Outlier Detection**: Analyze distributions and IQR boundaries.
4. **Exploratory Data Analysis (EDA)**: Target distribution analysis and correlation heatmap.

In [ ]:
missing_summary = df.isnull().sum()
print("Missing values per column:")
print(missing_summary[missing_summary > 0] if missing_summary.sum() > 0 else "✅ No missing values detected in dataset.")
if 'Default' in df.columns and 'LoanDefault' not in df.columns:
    df['LoanDefault'] = df['Default']
binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].replace({'Yes': 1, 'No': 0, 'yes': 1, 'no': 0, 'Y': 1, 'N': 0, 1: 1, 0: 0}).fillna(0).astype(int)
FEATURE_NAMES = [
    'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
    'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio',
    'HasMortgage', 'HasDependents'
]
X = df[FEATURE_NAMES]
y = df['LoanDefault'].astype(int)
print(f"Selected {len(FEATURE_NAMES)} features for modeling.")

In [ ]:
print("--- Outlier Detection (IQR Method) ---")
for col in ['Income', 'LoanAmount', 'CreditScore', 'DTIRatio', 'InterestRate']:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((X[col] < lower_bound) | (X[col] > upper_bound)).sum()
    print(f"{col:<16}: IQR={IQR:>8.2f} | Bounds=[{lower_bound:>8.2f}, {upper_bound:>8.2f}] | Outliers={outliers:>6,d} ({outliers/len(X)*100:.2f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
class_counts = y.value_counts()
sns.barplot(x=class_counts.index, y=class_counts.values, palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Loan Default Class Distribution (Target)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Loan Default (0 = Non-Default, 1 = Default)')
axes[0].set_ylabel('Number of Applicants')
for i, count in enumerate(class_counts.values):
    pct = count / len(y) * 100
    axes[0].text(i, count + (len(y)*0.01), f"{count:,} ({pct:.1f}%)", ha='center', fontweight='bold')
corr_matrix = df[FEATURE_NAMES + ['LoanDefault']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1], cbar_kws={'label': 'Pearson Correlation'})
axes[1].set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# ⚙️ Section 3: Model Creation (With Library & Scratch Implementation)

### 📐 Mathematical Foundation of Logistic Regression from Scratch:
1. **Hypothesis Function (Sigmoid):**
   $$z = X \mathbf{w} + b$$
   $$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$
2. **Binary Cross-Entropy Loss (Cost Function):**
   $$J(\mathbf{w}, b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$
3. **Gradients:**
   $$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} X^T (\hat{\mathbf{y}} - \mathbf{y})$$
   $$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$
4. **Parameter Updates:**
   $$\mathbf{w} := \mathbf{w} - \alpha \frac{\partial J}{\partial \mathbf{w}}, \quad b := b - \alpha \frac{\partial J}{\partial b}$$

In [ ]:
class CustomLogisticRegressionFromScratch:
    def __init__(self, learning_rate=0.05, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.loss_history = []
        y_arr = np.array(y, dtype=np.float64)
        X_arr = np.array(X, dtype=np.float64)
        for i in range(self.n_iterations):
            linear_model = np.dot(X_arr, self.weights) + self.bias
            y_predicted = self._sigmoid(linear_model)
            epsilon = 1e-15
            y_pred_clipped = np.clip(y_predicted, epsilon, 1 - epsilon)
            loss = -np.mean(y_arr * np.log(y_pred_clipped) + (1 - y_arr) * np.log(1 - y_pred_clipped))
            self.loss_history.append(loss)
            dw = (1 / n_samples) * np.dot(X_arr.T, (y_predicted - y_arr))
            db = (1 / n_samples) * np.sum(y_predicted - y_arr)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            if (i + 1) % 200 == 0 or i == 0:
                print(f"Iteration {i+1:>4d}/{self.n_iterations} | Loss: {loss:.5f}")
    def predict_proba(self, X):
        linear_model = np.dot(np.array(X), self.weights) + self.bias
        return self._sigmoid(linear_model)
    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)
        return (probs >= threshold).astype(int)
print("Custom Logistic Regression class created successfully without external ML libraries!")

---
# 📊 Section 4: Data Splitting, Scaling & Baseline Models

- **Stratified Split (80/20):** Balances class proportions across splits.
- **StandardScaler:** Fits on training set and saved to `models/scaler.pkl`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve, precision_recall_curve
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
os.makedirs('models', exist_ok=True)
joblib.dump(scaler, 'models/scaler.pkl')
print(f"Training Data: {X_train.shape[0]:,} samples")
print(f"Testing Data:  {X_test.shape[0]:,} samples")
print("Scaler successfully saved to models/scaler.pkl")

In [ ]:
model_metrics = {}
train_test_comparison = {}
def evaluate_and_record(name, model, X_tr, y_tr, X_te, y_te, train_time=0.0):
    y_train_pred = model.predict(X_tr)
    y_test_pred = model.predict(X_te)
    if hasattr(model, "predict_proba"):
        y_test_prob = model.predict_proba(X_te)
        if y_test_prob.ndim == 2:
            y_test_prob = y_test_prob[:, 1]
    elif hasattr(model, "decision_function"):
        y_test_prob = model.decision_function(X_te)
    else:
        y_test_prob = y_test_pred
    train_acc = accuracy_score(y_tr, y_train_pred)
    test_acc = accuracy_score(y_te, y_test_pred)
    prec = precision_score(y_te, y_test_pred, zero_division=0)
    rec = recall_score(y_te, y_test_pred)
    f1 = f1_score(y_te, y_test_pred)
    roc_auc = roc_auc_score(y_te, y_test_prob)
    model_metrics[name] = {
        "Accuracy": test_acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "ROC-AUC": roc_auc,
        "Time (s)": round(train_time, 2)
    }
    train_test_comparison[name] = {
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Overfit Gap": train_acc - test_acc
    }
    print(f"=== {name} ===")
    print(f"Train Acc: {train_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}% | Gap: {(train_acc - test_acc)*100:+.2f}%")
    print(f"Recall: {rec*100:.2f}% | Precision: {prec*100:.2f}% | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f} | Time: {train_time:.2f}s")
    return y_test_pred, y_test_prob

In [ ]:
print("--- 1. Training Scratch Custom Logistic Regression ---")
scratch_lr = CustomLogisticRegressionFromScratch(learning_rate=0.1, n_iterations=600)
t0 = time.time()
scratch_lr.fit(X_train_scaled, y_train)
scratch_time = time.time() - t0
scratch_pred = scratch_lr.predict(X_test_scaled)
scratch_prob = scratch_lr.predict_proba(X_test_scaled)
print(f"Scratch Accuracy: {accuracy_score(y_test, scratch_pred)*100:.2f}% | ROC-AUC: {roc_auc_score(y_test, scratch_prob):.4f}")
print("\n--- 2. Training Scikit-Learn Logistic Regression ---")
from sklearn.linear_model import LogisticRegression
t0 = time.time()
sk_lr = LogisticRegression(max_iter=1000, random_state=42)
sk_lr.fit(X_train_scaled, y_train)
sk_time = time.time() - t0
sk_pred, sk_prob = evaluate_and_record("Logistic Regression", sk_lr, X_train_scaled, y_train, X_test_scaled, y_test, train_time=sk_time)
joblib.dump(sk_lr, 'models/logistic_regression.pkl')

In [ ]:
from sklearn.tree import DecisionTreeClassifier
t0 = time.time()
dt_model = DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)
dt_model.fit(X_train, y_train)
dt_time = time.time() - t0
dt_pred, dt_prob = evaluate_and_record("Decision Tree", dt_model, X_train, y_train, X_test, y_test, train_time=dt_time)
joblib.dump(dt_model, 'models/decision_tree.pkl')

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
sample_size = min(30000, len(X_train))
knn_idx = np.random.RandomState(42).choice(len(X_train_scaled), sample_size, replace=False)
X_tr_knn = X_train_scaled[knn_idx]
y_tr_knn = y_train.iloc[knn_idx]
t0 = time.time()
knn_model = KNeighborsClassifier(n_neighbors=9, weights='distance', n_jobs=-1)
knn_model.fit(X_tr_knn, y_tr_knn)
knn_time = time.time() - t0
test_knn_idx = np.random.RandomState(42).choice(len(X_test_scaled), min(10000, len(X_test_scaled)), replace=False)
knn_pred, knn_prob = evaluate_and_record("K-Nearest Neighbors", knn_model, X_tr_knn, y_tr_knn, X_test_scaled[test_knn_idx], y_test.iloc[test_knn_idx], train_time=knn_time)
joblib.dump(knn_model, 'models/k_neighbours.pkl')

In [ ]:
from sklearn.ensemble import RandomForestClassifier
t0 = time.time()
rf_model = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_time = time.time() - t0
rf_pred, rf_prob = evaluate_and_record("Random Forest", rf_model, X_train, y_train, X_test, y_test, train_time=rf_time)
joblib.dump(rf_model, 'models/random_forest.pkl', compress=3)

---
# 🚀 Section 5: Advanced Models, Cross-Validation & Hyperparameter Tuning

1. **Gaussian Naive Bayes (GNB)**: Probabilistic classifier.
2. **Calibrated Support Vector Classifier (SVC)**: Maximum margin classification.
3. **Bagging Classifier**: Bootstrap aggregation.
4. **Histogram Gradient Boosting & AdaBoost**: Sequential boosting.
5. **Stratified 5-Fold Cross-Validation**: Stability validation.
6. **Hyperparameter Tuning (`GridSearchCV`)**: Tree depth optimization.

In [ ]:
from sklearn.naive_bayes import GaussianNB
t0 = time.time()
gnb_model = GaussianNB()
gnb_model.fit(X_train_scaled, y_train)
gnb_time = time.time() - t0
gnb_pred, gnb_prob = evaluate_and_record("Gaussian Naive Bayes", gnb_model, X_train_scaled, y_train, X_test_scaled, y_test, train_time=gnb_time)
joblib.dump(gnb_model, 'models/gaussian_nb.pkl')
joblib.dump(gnb_model, 'models/probabilistic.pkl')

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
t0 = time.time()
base_svc = LinearSVC(class_weight='balanced', random_state=42, max_iter=3000, dual=False)
svc_model = CalibratedClassifierCV(estimator=base_svc, cv=3)
svc_model.fit(X_train_scaled, y_train)
svc_time = time.time() - t0
svc_pred, svc_prob = evaluate_and_record("Support Vector Classification (SVC)", svc_model, X_train_scaled, y_train, X_test_scaled, y_test, train_time=svc_time)
joblib.dump(svc_model, 'models/svc.pkl')
joblib.dump(svc_model, 'models/svm.pkl')

In [ ]:
from sklearn.ensemble import BaggingClassifier
t0 = time.time()
base_dt = DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42)
bagging_model = BaggingClassifier(estimator=base_dt, n_estimators=60, random_state=42, n_jobs=-1)
bagging_model.fit(X_train, y_train)
bag_time = time.time() - t0
bag_pred, bag_prob = evaluate_and_record("Bagging Classifier", bagging_model, X_train, y_train, X_test, y_test, train_time=bag_time)
joblib.dump(bagging_model, 'models/bagging.pkl')

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, AdaBoostClassifier
t0 = time.time()
boosting_model = HistGradientBoostingClassifier(max_iter=160, class_weight='balanced', random_state=42, min_samples_leaf=20)
boosting_model.fit(X_train, y_train)
boost_time = time.time() - t0
boost_pred, boost_prob = evaluate_and_record("Gradient Boosting", boosting_model, X_train, y_train, X_test, y_test, train_time=boost_time)
joblib.dump(boosting_model, 'models/boosting.pkl')
joblib.dump(boosting_model, 'models/gradient_boosting.pkl')
t0 = time.time()
adaboost_model = AdaBoostClassifier(n_estimators=100, random_state=42)
adaboost_model.fit(X_train, y_train)
ada_time = time.time() - t0
ada_pred, ada_prob = evaluate_and_record("AdaBoost", adaboost_model, X_train, y_train, X_test, y_test, train_time=ada_time)
joblib.dump(adaboost_model, 'models/adaboost.pkl')

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_models = {
    "Logistic Regression": (sk_lr, X_train_scaled),
    "Gaussian Naive Bayes": (gnb_model, X_train_scaled),
    "Decision Tree": (dt_model, X_train),
    "Gradient Boosting": (boosting_model, X_train)
}
print("--- 5-Fold Stratified Cross-Validation (ROC-AUC) ---")
for name, (m, data) in cv_models.items():
    scores = cross_val_score(m, data, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f"{name:<24}: Mean AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")

---
# 📈 Section 6: Performance Visualizations & Diagnostics

In [ ]:
results_df = pd.DataFrame(model_metrics).T
results_df = results_df.sort_values(by="ROC_AUC", ascending=False)
joblib.dump(model_metrics, 'models/new_models_metrics.pkl')
print("📊 All Model Benchmark Results:")
display(results_df.style.format({
    'Accuracy': '{:.2%}',
    'Precision': '{:.2%}',
    'Recall': '{:.2%}',
    'F1 Score': '{:.4f}',
    'ROC-AUC': '{:.4f}',
    'Time (s)': '{:.2f}s'
}))

In [ ]:
plt.figure(figsize=(11, 7))
all_probs = {
    "Logistic Regression": sk_prob,
    "Decision Tree": dt_prob,
    "Random Forest": rf_prob,
    "Gaussian Naive Bayes": gnb_prob,
    "Calibrated SVC": svc_prob,
    "Bagging Classifier": bag_prob,
    "Gradient Boosting": boost_prob,
    "AdaBoost": ada_prob
}
for name, probs in all_probs.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Baseline (AUC = 0.500)', alpha=0.7)
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=12)
plt.title('ROC Curves - All Machine Learning Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models_to_plot = [
    ("Gradient Boosting", boost_pred),
    ("Random Forest", rf_pred),
    ("Calibrated SVC", svc_pred)
]
for ax, (name, pred) in zip(axes, models_to_plot):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Non-Default', 'Default'],
                yticklabels=['Non-Default', 'Default'])
    ax.set_title(f'Confusion Matrix: {name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()

In [ ]:
if hasattr(rf_model, "feature_importances_"):
    feat_importances = pd.Series(rf_model.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=True)
    plt.figure(figsize=(10, 6))
    feat_importances.plot(kind='barh', color='#16a085', edgecolor='black', alpha=0.85)
    plt.title('Feature Importance Ranking (Random Forest Classifier)', fontsize=14, fontweight='bold')
    plt.xlabel('Relative Importance Gini Score')
    plt.tight_layout()
    plt.show()

---
# 📦 Section 7: Export Summary & Web App Deployment

All trained model weights, scalers, and evaluation metrics are saved to the `models/` folder and consumed by `app.py`.

In [ ]:
print("📁 Current Contents of models/ folder:")
for fname in sorted(os.listdir('models')):
    fpath = os.path.join('models', fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  • models/{fname:<30} ({size_kb:>8.1f} KB)")